# Análisis inmobiliario: descripción inicial

El objetivo de esta primera etapa es conocer la estructura del dataset `secondary_market.csv` antes de tomar decisiones de limpieza o modelado.

**Importante:** en esta etapa no se elimina, transforma ni excluye ninguna columna. Trabajamos sobre una copia llamada `df_original`.

## Descripción del dataset

El archivo `secondary_market.csv` contiene publicaciones de inmuebles usados en venta. Cada fila representa una publicación inmobiliaria y cada columna describe una característica del inmueble, su ubicación, el edificio, la cercanía al transporte, las condiciones de publicación y sus precios.

El dataset original contiene 50.000 registros y 26 columnas. Incluye identificadores y fechas; ubicación geográfica (`district`, `okrug`, latitud y longitud); superficie y cantidad de ambientes; piso y antigüedad del edificio; tipo de construcción, altura de techo y renovación; estación y línea de metro; distancia al metro y al centro; tipo de vendedor; tasa hipotecaria; precio total (`price_rub`) y precio por metro cuadrado (`price_per_sqm`).

El objetivo posible del análisis es predecir `price_per_sqm`, porque permite comparar inmuebles de diferentes superficies. Si se utiliza ese objetivo, `price_rub` no debe usarse como predictor porque se calcula directamente a partir de `total_area` y `price_per_sqm`.

El archivo original se conserva sin modificaciones. Las transformaciones se realizan sobre copias de trabajo y se guardan en `data_sets_transformados`.

## 1. Carga y `describe` inicial

Primero cargamos el dataset y ejecutamos `describe(include="all")`. La transposición `.T` permite leer cada variable como una fila. Para las variables numéricas veremos estadísticos como media, desvío y cuartiles; para las categóricas veremos cantidad de valores, cantidad de categorías, categoría más frecuente y frecuencia.

In [1]:
from pathlib import Path

import pandas as pd

RUTA_DATASET = Path("data_sets/secondary_market.csv")
df_original = pd.read_csv(RUTA_DATASET)

print(f"Dataset cargado: {RUTA_DATASET.resolve()}")
print(f"Filas: {df_original.shape[0]} | Columnas: {df_original.shape[1]}")

print("Información general del dataset:")
df_original.info()

descripcion_inicial = df_original.describe(include="all").T
display(descripcion_inicial)

Dataset cargado: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets\secondary_market.csv
Filas: 50000 | Columnas: 26
Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        50000 non-null  str    
 1   date_posted               50000 non-null  str    
 2   district                  50000 non-null  str    
 3   okrug                     50000 non-null  str    
 4   lat                       50000 non-null  float64
 5   lon                       50000 non-null  float64
 6   total_area                50000 non-null  float64
 7   living_area               50000 non-null  float64
 8   kitchen_area              50000 non-null  float64
 9   rooms                     50000 non-null  int64  
 10  floor                     50000 non-null  int64  
 11  total_floor

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,50000,50000,S000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_posted,50000,2311,2024-02-26,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
district,50000,129,Begovoy,434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
okrug,50000,11,SVAO,6662,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,50000.0,NaN,NaN,NaN,55.712683,0.143998,55.23408,55.635415,55.74011,55.81749,56.02047
lon,50000.0,NaN,NaN,NaN,37.543657,0.180338,36.69923,37.46711,37.56584,37.65202,37.91986
total_area,50000.0,NaN,NaN,NaN,67.053158,17.309579,20.9,54.4,65.1,77.5,134.3
living_area,50000.0,NaN,NaN,NaN,41.883782,11.209233,12.6,33.7,40.6,48.5,89.2
kitchen_area,50000.0,NaN,NaN,NaN,11.24465,2.855538,6.0,9.2,11.1,13.1,24.8
rooms,50000.0,NaN,NaN,NaN,2.1349,1.023153,1.0,1.0,2.0,3.0,5.0


## Análisis del `describe`

El resultado inicial muestra 50.000 registros y 26 columnas, sin valores faltantes. Todavía no se elimina ninguna columna ni fila; las siguientes decisiones son hipótesis para validar durante la auditoría y el modelado.

### Análisis de `info()`

`info()` muestra que las 26 columnas tienen 50.000 valores no nulos. Por lo tanto, no se observan datos faltantes en esta primera revisión. El dataset contiene 10 variables de texto, 8 variables decimales, 7 variables enteras y 1 variable booleana.

`date_posted` aparece inicialmente como texto; no es un error de los datos, pero deberá convertirse a fecha para poder ordenar temporalmente, extraer año y mes, y construir correctamente la división de entrenamiento y prueba. Las variables de texto categóricas deberán codificarse recién en la etapa de preparación.

### Posibles columnas a excluir del modelo

- `id`: tiene un valor único por registro y funciona como identificador, no como característica del inmueble.
- `price_rub`: si el objetivo es `price_per_sqm`, debe excluirse para evitar fuga de información porque `price_rub = total_area × price_per_sqm`.
- `price_per_sqm`: será la variable objetivo, por lo que no debe utilizarse como predictor. No se elimina del dataset original.

### Variable predictora objetivo

La variable objetivo posible es `price_per_sqm`, porque permite comparar el valor relativo de inmuebles con superficies diferentes. El modelo intentará explicar ese valor a partir de ubicación, superficie, características del edificio, renovación, cercanía al metro y condiciones de publicación.

### Variables que debemos revisar

- `date_posted`: debe convertirse en variables temporales, como año y mes, antes de modelar.
- `district` y `okrug`: representan niveles geográficos relacionados; debemos evaluar si conservar ambos genera redundancia.
- `metro_station` y `metro_line`: una estación pertenece a una línea, por lo que podrían aportar información repetida.
- `total_area`, `living_area` y `kitchen_area`: pueden estar correlacionadas y generar problemas de multicolinealidad.
- `lat` y `lon`: pueden solaparse parcialmente con las variables geográficas categóricas, aunque aportan mayor detalle espacial.
- `building_year`: podría transformarse en antigüedad al momento de la publicación.
- `price_per_sqm`, `price_rub`, `metro_distance_min` y `to_center_km`: presentan valores extremos que deben auditarse antes de decidir si son casos reales o errores.

### Notas de interpretación

- Los `NaN` que aparecen en algunas celdas de `describe(include="all")` pueden indicar que el estadístico no aplica al tipo de variable. Por ejemplo, una variable categórica de texto no tiene media, desvío estándar ni cuartiles. Para confirmar si existen faltantes reales, se debe revisar `df_original.isna().sum()`; en este dataset, el `describe` muestra `count = 50.000` en todas las columnas.
- En `price_per_sqm`, el máximo (`1.239.100 RUB/m²`) presenta una diferencia muy grande respecto de la media (`195.625,67 RUB/m²`) y la mediana (`170.400 RUB/m²`). Por eso debe considerarse un posible outlier. Antes de eliminarlo, hay que revisar el inmueble correspondiente y verificar si el valor es real o un error de carga.

La próxima etapa será auditar dominios, duplicados, cardinalidades, correlaciones y valores extremos. Recién después decidiremos qué columnas excluir, transformar o conservar.

## Relación con datasets auxiliares

`district_prices_monthly.csv` está a nivel distrito-mes y `metro_stations.csv` a nivel estación. No comparten una clave directa, por lo que no se deben unir entre sí sin un mapeo espacial distrito-estación.

El panel mensual podría enriquecer las publicaciones usando distrito y mes, preferentemente con valores del mes anterior para evitar fuga de información. La tabla de estaciones puede validar o enriquecer las publicaciones por estación y línea, pero no es necesaria para el primer modelo. Por ahora se mantienen separadas.

## 2. Transformación de variables

Antes de eliminar columnas, transformamos las variables que puedan mejorar la interpretación del modelo. Trabajamos sobre una copia llamada `df_transformado` y conservamos `df_original` sin cambios.

### 2.1 Transformación de `date_posted`

En este primer bloque transformamos `date_posted` para obtener el año y el mes de publicación.

El año permitirá ordenar y separar temporalmente los datos. El mes puede capturar patrones estacionales del mercado inmobiliario.

In [3]:
# Creamos una copia para transformar sin modificar el dataset original.
df_transformado = df_original.copy()

# Convertimos la columna de texto a un tipo de fecha reconocido por pandas.
df_transformado["date_posted"] = pd.to_datetime(
    df_transformado["date_posted"],
    errors="coerce",
)

# Contamos las fechas que no pudieron convertirse para detectar errores de formato.
fechas_invalidas = df_transformado["date_posted"].isna().sum()

# Extraemos el año de cada fecha de publicación.
df_transformado["date_posted_year"] = df_transformado["date_posted"].dt.year

# Extraemos el número de mes de cada fecha de publicación.
df_transformado["date_posted_month"] = df_transformado["date_posted"].dt.month

# Definimos la carpeta donde guardaremos la versión transformada.
RUTA_DATASET_TRANSFORMADO = Path("data_sets_transformados")

# Creamos la carpeta si todavía no existe.
RUTA_DATASET_TRANSFORMADO.mkdir(parents=True, exist_ok=True)

# Definimos el nombre del nuevo archivo CSV.
RUTA_SALIDA = RUTA_DATASET_TRANSFORMADO / "secondary_market_date_transformado.csv"

# Guardamos todas las columnas originales más las columnas de año y mes.
df_transformado.to_csv(RUTA_SALIDA, index=False)

# Informamos si alguna fecha quedó inválida después de la conversión.
print(f"Fechas inválidas después de la conversión: {fechas_invalidas}")

# Informamos dónde quedó guardado el nuevo dataset.
print(f"Dataset transformado guardado en: {RUTA_SALIDA.resolve()}")

# Mostramos las columnas transformadas para verificar el resultado.
display(
    df_transformado[
        ["date_posted", "date_posted_year", "date_posted_month"]
    ].head()
)

Fechas inválidas después de la conversión: 0
Dataset transformado guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_transformado.csv


,date_posted,date_posted_year,date_posted_month
0,2024-11-23,2024,11
1,2025-08-13,2025,8
2,2022-05-06,2022,5
3,2025-04-08,2025,4
4,2023-07-16,2023,7


### Interpretación de la transformación

La fecha original se convirtió correctamente a formato fecha y se agregaron dos variables numéricas: año y mes. No se eliminaron registros ni columnas. Si `fechas_invalidas` es cero, todas las fechas pudieron convertirse; si fuera mayor que cero, habría que revisar esos registros antes de continuar.

### 2.2 Transformación de `building_year`

Trabajamos sobre el archivo transformado generado en el bloque anterior. Creamos `building_age` como la diferencia entre el año de publicación y el año de construcción. No eliminamos `building_year` ni corregimos automáticamente las antigüedades negativas; primero vamos a auditarlas.

In [4]:
# Definimos la ruta del archivo que ya contiene la transformación de date_posted.
RUTA_SALIDA = Path("data_sets_transformados/secondary_market_date_transformado.csv")

# Cargamos la versión transformada y convertimos nuevamente date_posted a fecha.
df_transformado = pd.read_csv(RUTA_SALIDA, parse_dates=["date_posted"])

# Calculamos la antigüedad aproximada al momento de la publicación.
df_transformado["building_age"] = (
    df_transformado["date_posted_year"]
    - df_transformado["building_year"]
)

# Contamos las antigüedades negativas para detectar posibles inconsistencias.
antiguedades_negativas = (df_transformado["building_age"] < 0).sum()

# Guardamos la nueva columna en el mismo archivo transformado.
df_transformado.to_csv(RUTA_SALIDA, index=False)

# Informamos el resultado de la auditoría inicial de antigüedad.
print(f"Antigüedades negativas: {antiguedades_negativas}")
print(f"Dataset actualizado: {RUTA_SALIDA.resolve()}")

# Mostramos las columnas involucradas para verificar la transformación.
display(
    df_transformado[
        ["date_posted", "building_year", "building_age"]
    ].head()
)

Antigüedades negativas: 1271
Dataset actualizado: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_transformado.csv


,date_posted,building_year,building_age
0,2024-11-23,1969,55
1,2025-08-13,2016,9
2,2022-05-06,2016,6
3,2025-04-08,2012,13
4,2023-07-16,2002,21


### Interpretación de `building_age`

La nueva variable representa la antigüedad aproximada del inmueble al publicarse. Se obtuvieron 48.729 valores no negativos, 655 inmuebles con antigüedad 0 y 1.271 valores negativos, aproximadamente el 2,5% del dataset. El rango va de -4 a 91 años.

Los valores negativos indican que `building_year` es posterior a `date_posted_year`. No se eliminan todavía: deben revisarse como posibles inconsistencias o publicaciones de inmuebles aún en construcción.

### 2.3 Análisis de `price_per_sqm`

Analizamos `price_per_sqm` porque es la variable objetivo posible del modelo. En esta etapa revisamos su distribución y sus valores extremos, pero no eliminamos registros ni modificamos la variable.

In [ ]:
# Definimos la ruta del dataset que contiene las transformaciones anteriores.
RUTA_SALIDA = Path("data_sets_transformados/secondary_market_date_transformado.csv")

# Cargamos el dataset transformado sin modificar sus registros.
df_analisis = pd.read_csv(RUTA_SALIDA, parse_dates=["date_posted"])

# Seleccionamos la variable objetivo y excluimos valores nulos solo para calcular las estadísticas.
objetivo = df_analisis["price_per_sqm"].dropna()

# Calculamos los cuartiles para identificar valores extremos mediante el rango intercuartílico.
q1 = objetivo.quantile(0.25)
q3 = objetivo.quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Contamos los valores que quedan fuera de los límites calculados.
cantidad_outliers_inferiores = (objetivo < limite_inferior).sum()
cantidad_outliers_superiores = (objetivo > limite_superior).sum()

# Construimos un resumen con las medidas principales de la variable objetivo.
resumen_price_per_sqm = pd.DataFrame({
    "métrica": [
        "registros",
        "valores faltantes",
        "media",
        "mediana",
        "desvío estándar",
        "mínimo",
        "máximo",
        "asimetría",
        "outliers inferiores",
        "outliers superiores"
    ],
    "valor": [
        objetivo.size,
        df_analisis["price_per_sqm"].isna().sum(),
        objetivo.mean(),
        objetivo.median(),
        objetivo.std(),
        objetivo.min(),
        objetivo.max(),
        objetivo.skew(),
        cantidad_outliers_inferiores,
        cantidad_outliers_superiores
    ]
})

# Mostramos el resumen para evaluar la distribución y los valores extremos.
display(resumen_price_per_sqm)

# Identificamos el registro con el mayor precio por metro cuadrado para auditarlo.
indice_maximo = df_analisis["price_per_sqm"].idxmax()
display(df_analisis.loc[[indice_maximo]])

### Interpretación de `price_per_sqm`

No se observan valores faltantes. La media es 195.625,67 RUB/m² y la mediana es 170.400 RUB/m²; junto con una asimetría positiva de 1,69, esto indica que algunos precios altos elevan la media. El máximo es 1.239.100 RUB/m² y el criterio IQR detecta 2.696 outliers superiores, sin outliers inferiores. Estos registros deben auditarse antes de decidir si son errores o propiedades reales; no se eliminan automáticamente.

### 2.4 Análisis de `district` y `okrug`

Estas variables son categóricas nominales: sus valores no representan una escala numérica. Primero revisamos su cardinalidad, frecuencia y relación jerárquica. Para el modelado, no se deben convertir directamente a números porque eso crearía un orden artificial; se recomienda aplicar one-hot encoding durante la preparación del modelo.

In [5]:
# Definimos la ruta del dataset transformado sin modificar el archivo original.
RUTA_SALIDA = Path("data_sets_transformados/secondary_market_date_transformado.csv")

# Cargamos las variables categóricas para auditarlas antes de transformarlas.
df_categoricas = pd.read_csv(RUTA_SALIDA)

# Indicamos las columnas geográficas que vamos a analizar.
columnas_geograficas = ["district", "okrug"]

# Calculamos cantidad de categorías, faltantes y frecuencia de la categoría más común.
resumen_geografico = pd.DataFrame({
    "variable": columnas_geograficas,
    "categorías": [df_categoricas[columna].nunique(dropna=False) for columna in columnas_geograficas],
    "faltantes": [df_categoricas[columna].isna().sum() for columna in columnas_geograficas],
    "frecuencia_máxima": [df_categoricas[columna].value_counts(dropna=False).iloc[0] for columna in columnas_geograficas]
})

# Mostramos el resumen de las dos variables categóricas.
display(resumen_geografico)

# Contamos cuántos distritos diferentes pertenecen a cada okrug.
distritos_por_okrug = (
    df_categoricas.groupby("okrug", dropna=False)["district"]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

# Verificamos cuántos okrugs distintos tiene asociado cada distrito.
okrugs_por_distrito = (
    df_categoricas.groupby("district", dropna=False)["okrug"]
    .nunique(dropna=False)
)

# Mostramos la consistencia de la relación jerárquica entre distrito y okrug.
print(f"Máximo de okrugs por distrito: {okrugs_por_distrito.max()}")
print(f"Máximo de distritos por okrug: {distritos_por_okrug.max()}")
print(f"Distritos asociados a más de un okrug: {(okrugs_por_distrito > 1).sum()}")
display(distritos_por_okrug.to_frame("cantidad_de_distritos"))

# Mostramos las combinaciones únicas para detectar relaciones geográficas inesperadas.
display(
    df_categoricas[["district", "okrug"]]
    .drop_duplicates()
    .sort_values(["okrug", "district"])
)

,variable,categorías,faltantes,frecuencia_máxima
0,district,129,0,434
1,okrug,11,0,6662


Máximo de okrugs por distrito: 1
Máximo de distritos por okrug: 17
Distritos asociados a más de un okrug: 0


,cantidad_de_distritos
okrug,
SVAO,17
VAO,16
YuAO,16
SAO,15
YuZAO,12
ZAO,12
NAO,11
CAO,10
TAO,10


,district,okrug
181,Arbat,CAO
33,Basmanny,CAO
188,Khamovniki,CAO
66,Krasnoselsky,CAO
344,Meshchansky,CAO
...,...,...
131,Ramenki,ZAO
61,Solntsevo,ZAO
101,Troparyovo-Nikulino,ZAO
510,Vnukovo,ZAO


### Interpretación de `district` y `okrug`

`district` tiene 129 categorías y `okrug` tiene 11; ninguna de las dos presenta faltantes. Cada distrito pertenece a un único `okrug`, mientras que un `okrug` agrupa varios distritos. La relación es consistente y jerárquica.

Para el primer modelo conviene aplicar one-hot encoding a `district`, porque aporta mayor detalle geográfico, y dejar `okrug` como alternativa agregada. No se recomienda usar ambas variables al mismo tiempo en el primer modelo porque contienen información geográfica redundante. Las columnas originales se conservan para la auditoría.

### 2.5 One-hot encoding de `district`

Trabajamos sobre el dataset transformado y conservamos las columnas originales `district` y `okrug`. Agregamos una columna binaria por cada categoría de `district`. `okrug` queda disponible sin modificaciones para auditoría y para decidir más adelante si se incorpora al modelo.

In [6]:
# Definimos la ruta del dataset transformado que contiene building_age.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_transformado.csv")

# Definimos la ruta de la nueva versión con las columnas one-hot.
RUTA_ONE_HOT = Path("data_sets_transformados/secondary_market_date_district_onehot_transformado.csv")

# Cargamos el dataset transformado sin modificar el archivo de entrada.
df_one_hot = pd.read_csv(RUTA_ENTRADA)

# Normalizamos espacios al inicio y al final de los nombres de distrito en la copia de trabajo.
df_one_hot["district"] = df_one_hot["district"].astype("string").str.strip()

# Creamos una columna binaria por cada categoría de district sin asignar un orden numérico artificial.
district_one_hot = pd.get_dummies(
    df_one_hot["district"],
    prefix="district",
    dtype="int8"
)

# Agregamos las columnas codificadas y conservamos district y okrug originales.
df_one_hot = pd.concat([df_one_hot, district_one_hot], axis=1)

# Verificamos que cada registro tenga exactamente un distrito activo.
cantidad_district_activo = district_one_hot.sum(axis=1).value_counts().to_dict()

# Guardamos la nueva versión preparada para la etapa de modelado.
df_one_hot.to_csv(RUTA_ONE_HOT, index=False)

# Informamos el resultado de la transformación.
print(f"Columnas one-hot creadas: {district_one_hot.shape[1]}")
print(f"Distribución de cantidad de categorías activas por registro: {cantidad_district_activo}")
print(f"Dataset guardado en: {RUTA_ONE_HOT.resolve()}")

# Mostramos las columnas originales y algunas columnas codificadas para verificar el resultado.
display(
    df_one_hot[["district", "okrug", *district_one_hot.columns[:5]]].head()
)

Columnas one-hot creadas: 129
Distribución de cantidad de categorías activas por registro: {1: 50000}
Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_district_onehot_transformado.csv


,district,okrug,district_Aeroport,district_Akademichesky,district_Alekseevsky,district_Altufyevsky,district_Arbat
0,Begovoy,SAO,0,0,0,0,0
1,Sokolniki,VAO,0,0,0,0,0
2,Sosenskoye,NAO,0,0,0,0,0
3,Sviblovo,SVAO,0,0,0,0,0
4,Begovoy,SAO,0,0,0,0,0


### Interpretación de la transformación

Se conserva `okrug` y se agregan 129 columnas binarias correspondientes a `district`. Cada fila debe tener exactamente una categoría de distrito activa. Para el modelado se utilizarán las columnas one-hot y se excluirá la columna de texto `district`; `okrug` queda conservado para auditoría y no se incorpora todavía como predictor.

### 2.6 Auditoría de escritura de `district`

Revisamos diferencias de mayúsculas, espacios exteriores y nombres muy similares. Estas comprobaciones permiten detectar duplicados producidos por errores de escritura, aunque la confirmación definitiva requeriría comparar contra un catálogo geográfico oficial.

In [ ]:
# Importamos una herramienta de la biblioteca estándar para comparar nombres parecidos.
from difflib import SequenceMatcher

# Cargamos el dataset transformado antes del one-hot encoding.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_transformado.csv")
df_auditoria_district = pd.read_csv(RUTA_ENTRADA)

# Obtenemos los nombres únicos de district sin valores faltantes.
districts = sorted(df_auditoria_district["district"].dropna().astype(str).unique())

# Normalizamos espacios y mayúsculas para detectar categorías duplicadas por escritura.
districts_normalizados = pd.Series(districts).str.strip().str.casefold()
duplicados_normalizados = (
    pd.DataFrame({"original": districts, "normalizado": districts_normalizados})
    .groupby("normalizado")["original"]
    .agg(list)
)
duplicados_normalizados = duplicados_normalizados[duplicados_normalizados.str.len() > 1]

# Detectamos nombres con espacios al inicio o al final.
nombres_con_espacios = [nombre for nombre in districts if nombre != nombre.strip()]

# Buscamos pares de nombres muy similares que podrían ser errores de escritura.
pares_similares = [
    {
        "district_a": district_a,
        "district_b": district_b,
        "similitud": round(SequenceMatcher(None, district_a.casefold(), district_b.casefold()).ratio(), 3)
    }
    for posicion, district_a in enumerate(districts)
    for district_b in districts[posicion + 1:]
    if SequenceMatcher(None, district_a.casefold(), district_b.casefold()).ratio() >= 0.88
]

# Mostramos los resultados de la auditoría automática.
print(f"Categorías revisadas: {len(districts)}")
print(f"Duplicados por normalización: {duplicados_normalizados.to_dict()}")
print(f"Nombres con espacios exteriores: {nombres_con_espacios}")
display(pd.DataFrame(pares_similares))

### Interpretación de la auditoría

No se detectaron duplicados por mayúsculas o espacios, ni pares de nombres sospechosamente similares entre las 129 categorías. Por lo tanto, no se corrige ni elimina ningún distrito automáticamente. Para validar la ortografía oficial de nombres como transliteraciones geográficas, sería necesario contrastarlos con un catálogo externo confiable.

### 2.7 Análisis de `building_type`

`building_type` describe el tipo constructivo del inmueble. Es una variable categórica nominal, por lo que no se debe convertir a números con un orden artificial. Primero revisamos sus categorías y frecuencias; luego agregamos una columna binaria por categoría mediante one-hot encoding.

In [7]:
# Definimos la ruta del dataset que ya contiene el one-hot encoding de district.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_district_onehot_transformado.csv")

# Definimos la ruta de la nueva versión con building_type codificado.
RUTA_BUILDING_TYPE = Path("data_sets_transformados/secondary_market_date_district_building_type_onehot_transformado.csv")

# Cargamos el dataset preparado sin modificar el archivo de entrada.
df_building_type = pd.read_csv(RUTA_ENTRADA)

# Normalizamos espacios en la copia de trabajo para evitar categorías duplicadas por formato.
df_building_type["building_type"] = df_building_type["building_type"].astype("string").str.strip()

# Creamos una columna binaria por cada tipo de construcción.
building_type_one_hot = pd.get_dummies(
    df_building_type["building_type"],
    prefix="building_type",
    dtype="int8"
)

# Agregamos las columnas codificadas y conservamos building_type original.
df_building_type = pd.concat([df_building_type, building_type_one_hot], axis=1)

# Verificamos que cada registro tenga exactamente un tipo de construcción activo.
cantidad_building_type_activo = building_type_one_hot.sum(axis=1).value_counts().to_dict()

# Guardamos la nueva versión preparada para el modelado.
df_building_type.to_csv(RUTA_BUILDING_TYPE, index=False)

# Informamos el resultado de la transformación.
print(f"Categorías de building_type: {building_type_one_hot.shape[1]}")
print(f"Distribución de categorías activas por registro: {cantidad_building_type_activo}")
print(f"Dataset guardado en: {RUTA_BUILDING_TYPE.resolve()}")

# Mostramos la distribución original de la variable.
display(df_building_type["building_type"].value_counts().to_frame("cantidad"))

Categorías de building_type: 6
Distribución de categorías activas por registro: {1: 50000}
Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_district_building_type_onehot_transformado.csv


,cantidad
building_type,
panel,10143
monolith,9997
modern_panel,9981
brick,8836
khrushchev,7528
stalin,3515


### Interpretación de `building_type`

La variable tiene 6 categorías, no presenta valores faltantes y no se detectan diferencias de escritura. Como es una variable nominal, se aplica one-hot encoding. Cada registro conserva su categoría original y tiene exactamente una columna binaria de tipo de construcción activa.

### 2.8 Transformación binaria de `has_balcony`

`has_balcony` ya contiene valores booleanos (`True` y `False`) y no presenta faltantes. Creamos `has_balcony_binary` con valores numéricos `1` y `0`, que son más adecuados para los modelos, sin eliminar la columna original.

In [8]:
# Definimos la ruta del dataset que contiene las transformaciones anteriores.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_district_building_type_onehot_transformado.csv")

# Definimos la ruta de la nueva versión con has_balcony en formato numérico.
RUTA_BALCONY = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_transformado.csv")

# Cargamos el dataset transformado sin modificar el archivo de entrada.
df_balcony = pd.read_csv(RUTA_ENTRADA)

# Convertimos True en 1 y False en 0 usando un tipo entero pequeño.
has_balcony_binary = df_balcony["has_balcony"].astype("int8").rename("has_balcony_binary")

# Agregamos la nueva columna en bloque para evitar fragmentar el DataFrame.
df_balcony = pd.concat([df_balcony, has_balcony_binary], axis=1)

# Verificamos la distribución de la nueva variable binaria.
distribucion_balcony = df_balcony["has_balcony_binary"].value_counts().sort_index()

# Guardamos la nueva versión sin eliminar la columna has_balcony original.
df_balcony.to_csv(RUTA_BALCONY, index=False)

# Informamos el resultado de la transformación.
print(f"Valores faltantes en has_balcony: {df_balcony['has_balcony'].isna().sum()}")
print(f"Distribución de has_balcony_binary: {distribucion_balcony.to_dict()}")
print(f"Dataset guardado en: {RUTA_BALCONY.resolve()}")

# Mostramos ambas columnas para verificar la correspondencia entre booleano y binario.
display(df_balcony[["has_balcony", "has_balcony_binary"]].head())

Valores faltantes en has_balcony: 0
Distribución de has_balcony_binary: {0: 12510, 1: 37490}
Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_district_building_type_balcony_transformado.csv


,has_balcony,has_balcony_binary
0,True,1
1,True,1
2,True,1
3,True,1
4,True,1


### Interpretación de `has_balcony`

La variable original no tenía faltantes y solo presentaba dos valores: `True` en 37.490 registros y `False` en 12.510. La nueva variable `has_balcony_binary` representa la misma información como `1` y `0`; para el modelado se utilizará la columna binaria y la original se conservará para auditoría.

### 2.9 One-hot encoding de `seller_type` y `renovation`

Ambas variables son categóricas nominales y no presentan valores faltantes. Creamos una columna binaria por cada categoría, conservando las columnas originales para auditoría.

In [9]:
# Definimos la ruta del dataset que contiene las transformaciones anteriores.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_transformado.csv")

# Definimos la ruta de la versión con seller_type y renovation codificadas.
RUTA_CATEGORICAS = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_seller_renovation_onehot_transformado.csv")

# Cargamos el dataset transformado sin modificar el archivo de entrada.
df_categoricas = pd.read_csv(RUTA_ENTRADA)

# Normalizamos espacios en ambas variables antes de crear las categorías binarias.
df_categoricas["seller_type"] = df_categoricas["seller_type"].astype("string").str.strip()
df_categoricas["renovation"] = df_categoricas["renovation"].astype("string").str.strip()

# Creamos las columnas binarias de tipo de vendedor.
seller_type_one_hot = pd.get_dummies(
    df_categoricas["seller_type"],
    prefix="seller_type",
    dtype="int8"
)

# Creamos las columnas binarias de tipo de renovación.
renovation_one_hot = pd.get_dummies(
    df_categoricas["renovation"],
    prefix="renovation",
    dtype="int8"
)

# Agregamos ambas codificaciones y conservamos las columnas originales.
df_categoricas = pd.concat(
    [df_categoricas, seller_type_one_hot, renovation_one_hot],
    axis=1
)

# Verificamos que cada registro tenga una categoría activa en cada grupo.
seller_activo = seller_type_one_hot.sum(axis=1).value_counts().to_dict()
renovation_activo = renovation_one_hot.sum(axis=1).value_counts().to_dict()

# Guardamos la nueva versión del dataset.
df_categoricas.to_csv(RUTA_CATEGORICAS, index=False)

# Informamos las columnas creadas y la validación de categorías activas.
print(f"Columnas seller_type creadas: {seller_type_one_hot.shape[1]}")
print(f"Columnas renovation creadas: {renovation_one_hot.shape[1]}")
print(f"Categorías seller_type activas por fila: {seller_activo}")
print(f"Categorías renovation activas por fila: {renovation_activo}")
print(f"Dataset guardado en: {RUTA_CATEGORICAS.resolve()}")

# Mostramos las variables originales y sus columnas codificadas.
display(
    df_categoricas[
        ["seller_type", "renovation", *seller_type_one_hot.columns, *renovation_one_hot.columns]
    ].head()
)

Columnas seller_type creadas: 2
Columnas renovation creadas: 4
Categorías seller_type activas por fila: {1: 50000}
Categorías renovation activas por fila: {1: 50000}
Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_district_building_type_balcony_seller_renovation_onehot_transformado.csv


,seller_type,renovation,seller_type_agency,seller_type_owner,renovation_cosmetic,renovation_designer,renovation_euro,renovation_no_renovation
0,agency,cosmetic,1,0,1,0,0,0
1,owner,no_renovation,0,1,0,0,0,1
2,owner,cosmetic,0,1,1,0,0,0
3,agency,euro,1,0,0,0,1,0
4,agency,cosmetic,1,0,1,0,0,0


### Interpretación de la transformación

`seller_type` tiene 2 categorías (`agency` y `owner`) y `renovation` tiene 4 (`cosmetic`, `euro`, `no_renovation` y `designer`). No hay valores faltantes. Se crearon 6 columnas binarias en total; la matriz final del modelo deberá eliminar una categoría de cada grupo o utilizar `OneHotEncoder(drop="first")` para evitar colinealidad con el intercepto.

### 2.10 Análisis de las variables del metro

Analizamos conjuntamente `metro_station`, `metro_line`, `metro_distance_min` y `metro_distance_type` para identificar redundancias y entender cómo debe interpretarse la distancia.

In [10]:
# Definimos la ruta del dataset que contiene las transformaciones anteriores.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_seller_renovation_onehot_transformado.csv")

# Cargamos el dataset transformado sin modificar el archivo de entrada.
df_metro = pd.read_csv(RUTA_ENTRADA)

# Indicamos las columnas relacionadas con el acceso al metro.
columnas_metro = ["metro_station", "metro_line", "metro_distance_min", "metro_distance_type"]

# Resumimos tipos, cantidad de categorías y valores faltantes.
resumen_metro = pd.DataFrame({
    "variable": columnas_metro,
    "tipo": [str(df_metro[columna].dtype) for columna in columnas_metro],
    "categorías": [df_metro[columna].nunique(dropna=False) for columna in columnas_metro],
    "faltantes": [df_metro[columna].isna().sum() for columna in columnas_metro]
})

# Mostramos el resumen de las variables del metro.
display(resumen_metro)

# Contamos cuántas líneas distintas aparecen asociadas a cada estación.
lineas_por_estacion = (
    df_metro.groupby("metro_station", dropna=False)["metro_line"]
    .nunique(dropna=False)
 )

# Contamos cuántas estaciones diferentes tiene cada línea.
estaciones_por_linea = (
    df_metro.groupby("metro_line", dropna=False)["metro_station"]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

# Calculamos estadísticas de distancia separando si el acceso es caminando o por transporte.
distancia_por_tipo = (
    df_metro.groupby("metro_distance_type", dropna=False)["metro_distance_min"]
    .agg(["count", "mean", "median", "min", "max"])
)

# Mostramos las relaciones y la distribución de las distancias.
print(f"Estaciones asociadas a más de una línea: {(lineas_por_estacion > 1).sum()}")
display(estaciones_por_linea.to_frame("cantidad_de_estaciones"))
display(distancia_por_tipo)

# Mostramos las combinaciones estación-línea para verificar su consistencia.
display(
    df_metro[["metro_station", "metro_line"]]
    .drop_duplicates()
    .sort_values(["metro_line", "metro_station"])
)

,variable,tipo,categorías,faltantes
0,metro_station,str,104,0
1,metro_line,str,12,0
2,metro_distance_min,int64,317,0
3,metro_distance_type,str,2,0


Estaciones asociadas a más de una línea: 0


,cantidad_de_estaciones
metro_line,
Serpukhovsko-Timiryazevskaya,13
Sokolnicheskaya,13
Arbatsko-Pokrovskaya,12
Kaluzhsko-Rizhskaya,12
Filyovskaya,10
Koltsevaya,10
Zamoskvoretskaya,9
Lyublinsko-Dmitrovskaya,8
Tagansko-Krasnopresnenskaya,8


,count,mean,median,min,max
metro_distance_type,,,,,
transport,39771,68.061930,51.0,12,328
walk,10229,14.302864,12.0,2,39


,metro_station,metro_line
168,Elektrozavodskaya,Arbatsko-Pokrovskaya
71,Molodyozhnaya,Arbatsko-Pokrovskaya
208,Park Pobedy,Arbatsko-Pokrovskaya
152,Partizanskaya,Arbatsko-Pokrovskaya
89,Pervomayskaya,Arbatsko-Pokrovskaya
...,...,...
16,Rechnoy Vokzal,Zamoskvoretskaya
0,Sokol,Zamoskvoretskaya
83,Teatralnaya,Zamoskvoretskaya
22,Tverskaya,Zamoskvoretskaya


### Interpretación de las variables del metro

Las cuatro variables no tienen faltantes. Hay 104 estaciones y 12 líneas, y cada estación aparece asociada a una sola línea; por eso `metro_station` contiene la información de `metro_line` con mayor detalle y no conviene usar ambas en el primer modelo.

`metro_distance_type` distingue entre `walk` y `transport`. La distancia media es aproximadamente 14,3 minutos caminando y 68,1 minutos usando transporte. Por lo tanto, `metro_distance_min` no debe analizarse aislada: su significado depende de `metro_distance_type`. Se recomienda conservar ambas variables, codificar `metro_distance_type` y usar una sola entre `metro_station` y `metro_line`; la opción más informativa es `metro_station`.

### 2.11 One-hot encoding de `metro_station` y `metro_distance_type`

Codificamos las dos variables categóricas seleccionadas para el modelado. Conservamos `metro_station`, `metro_line` y `metro_distance_type` originales; `metro_line` no se codifica porque es redundante con `metro_station`.

In [11]:
# Definimos la ruta del dataset que contiene las transformaciones anteriores.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_seller_renovation_onehot_transformado.csv")

# Definimos la ruta de la nueva versión con las variables del metro codificadas.
RUTA_METRO = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_seller_renovation_metro_onehot_transformado.csv")

# Cargamos el dataset transformado sin modificar el archivo de entrada.
df_metro_one_hot = pd.read_csv(RUTA_ENTRADA)

# Normalizamos espacios en las variables categóricas antes de crear las columnas binarias.
df_metro_one_hot["metro_station"] = df_metro_one_hot["metro_station"].astype("string").str.strip()
df_metro_one_hot["metro_distance_type"] = df_metro_one_hot["metro_distance_type"].astype("string").str.strip()

# Creamos una columna binaria por cada estación.
metro_station_one_hot = pd.get_dummies(
    df_metro_one_hot["metro_station"],
    prefix="metro_station",
    dtype="int8"
)

# Creamos una columna binaria por cada tipo de distancia.
metro_distance_type_one_hot = pd.get_dummies(
    df_metro_one_hot["metro_distance_type"],
    prefix="metro_distance_type",
    dtype="int8"
)

# Agregamos ambas codificaciones y conservamos las columnas originales.
df_metro_one_hot = pd.concat(
    [df_metro_one_hot, metro_station_one_hot, metro_distance_type_one_hot],
    axis=1
)

# Verificamos que cada registro tenga una estación y un tipo de distancia activos.
estacion_activa = metro_station_one_hot.sum(axis=1).value_counts().to_dict()
tipo_distancia_activo = metro_distance_type_one_hot.sum(axis=1).value_counts().to_dict()

# Guardamos la nueva versión del dataset.
df_metro_one_hot.to_csv(RUTA_METRO, index=False)

# Informamos las columnas creadas y la validación de categorías activas.
print(f"Columnas de metro_station creadas: {metro_station_one_hot.shape[1]}")
print(f"Columnas de metro_distance_type creadas: {metro_distance_type_one_hot.shape[1]}")
print(f"Estaciones activas por fila: {estacion_activa}")
print(f"Tipos de distancia activos por fila: {tipo_distancia_activo}")
print(f"Dataset guardado en: {RUTA_METRO.resolve()}")

# Mostramos las variables originales y algunas columnas codificadas.
display(
    df_metro_one_hot[
        ["metro_station", "metro_line", "metro_distance_type", *metro_station_one_hot.columns[:5], *metro_distance_type_one_hot.columns]
    ].head()
)

Columnas de metro_station creadas: 104
Columnas de metro_distance_type creadas: 2
Estaciones activas por fila: {1: 50000}
Tipos de distancia activos por fila: {1: 50000}
Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_date_district_building_type_balcony_seller_renovation_metro_onehot_transformado.csv


,metro_station,metro_line,metro_distance_type,metro_station_Aleksandrovsky Sad,metro_station_Arbatskaya (F),metro_station_Bagrationovskaya,metro_station_Begovaya,metro_station_Belorusskaya,metro_distance_type_transport,metro_distance_type_walk
0,Sokol,Zamoskvoretskaya,transport,0,0,0,0,0,1,0
1,Sokolniki,Sokolnicheskaya,walk,0,0,0,0,0,0,1
2,Polyanka,Serpukhovsko-Timiryazevskaya,transport,0,0,0,0,0,1,0
3,Sviblovo,Kaluzhsko-Rizhskaya,walk,0,0,0,0,0,0,1
4,Sapyornoye Pole,Bolshaya Koltsevaya,walk,0,0,0,0,0,0,1


### Interpretación de la transformación

Se crearon 104 columnas binarias para `metro_station` y 2 para `metro_distance_type`. Cada registro tiene una estación y un tipo de distancia activos. `metro_line` se conserva, pero no se codifica porque queda representada por la estación; para el modelo final se deberá aplicar `drop="first"` para evitar colinealidad con el intercepto.

### 2.12 Eliminación de columnas no utilizadas

Creamos una versión final para el modelado eliminando las columnas originales que ya fueron reemplazadas por transformaciones o que no aportan información predictiva directa. También excluimos `price_rub` para evitar fuga de información hacia `price_per_sqm`. Los archivos anteriores se mantienen intactos como respaldo y trazabilidad.

**Nota:** reejecutar las celdas desde esta sección para actualizar las salidas guardadas de la revisión final y de las primeras 14 columnas.

In [20]:
# Definimos la ruta del dataset con todas las transformaciones realizadas.
RUTA_ENTRADA = Path("data_sets_transformados/secondary_market_date_district_building_type_balcony_seller_renovation_metro_onehot_transformado.csv")

# Definimos la ruta del dataset final para el modelado.
RUTA_FINAL = Path("data_sets_transformados/secondary_market_modelo_final.csv")

# Cargamos la versión más completa sin modificarla.
df_final = pd.read_csv(RUTA_ENTRADA)

# Indicamos las columnas originales que no se utilizarán como predictores.
columnas_a_eliminar = [
    "id",
    "date_posted",
    "district",
    "okrug",
    "building_year",
    "building_type",
    "has_balcony",
    "renovation",
    "seller_type",
    "metro_line",
    "metro_station",
    "metro_distance_type",
    "price_rub"
]

# Eliminamos únicamente las columnas solicitadas y conservamos las variables transformadas.
df_final = df_final.drop(columns=columnas_a_eliminar)

# Verificamos que ninguna columna solicitada permanezca en el dataset final.
columnas_pendientes = sorted(set(columnas_a_eliminar).intersection(df_final.columns))

# Guardamos el dataset final sin sobrescribir las versiones anteriores.
df_final.to_csv(RUTA_FINAL, index=False)

# Informamos el resultado de la eliminación.
print(f"Columnas pendientes de eliminar: {columnas_pendientes}")
print(f"Filas: {df_final.shape[0]} | Columnas: {df_final.shape[1]}")
print(f"Dataset final guardado en: {RUTA_FINAL.resolve()}")

# Mostramos las primeras columnas del dataset final para verificar la estructura.
display(df_final.head())

Columnas pendientes de eliminar: []
Filas: 50000 | Columnas: 264
Dataset final guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\data_sets_transformados\secondary_market_modelo_final.csv


,lat,lon,total_area,living_area,kitchen_area,rooms,floor,total_floors,ceiling_height,metro_distance_min,...,metro_station_Ulitsa Podbelskogo,metro_station_Universitet,metro_station_VDNKh,metro_station_Voykovskaya,metro_station_Vystavochnaya,metro_station_Yasenevo,metro_station_Yugo-Zapadnaya,metro_station_Yuzhnaya,metro_distance_type_transport,metro_distance_type_walk
0,55.79533,37.55519,71.0,46.5,6.1,2,19,25,2.5,18,...,0,0,0,0,0,0,0,0,1,0
1,55.79695,37.68875,79.6,52.8,10.8,3,2,25,2.5,7,...,0,0,0,0,0,0,0,0,0,1
2,55.54901,37.43118,74.0,42.2,8.9,2,6,9,3.0,102,...,0,0,0,0,0,0,0,0,1,0
3,55.86462,37.61471,52.3,34.0,6.0,1,5,22,2.5,9,...,0,0,0,0,0,0,0,0,0,1
4,55.79429,37.54631,62.2,38.3,13.1,2,12,12,2.5,10,...,0,0,0,0,0,0,0,0,0,1


### Interpretación de la eliminación

Se eliminan las 12 columnas originales solicitadas y `price_rub`, para evitar fuga de información al predecir `price_per_sqm`. Se conservan las variables derivadas (`date_posted_year`, `date_posted_month`, `building_age`, `has_balcony_binary`) y las columnas one-hot. La fecha original deja de estar disponible, pero el año y el mes permiten mantener información temporal para el análisis.

## 2.13 Revisión final del dataset

Cargamos el dataset final y repetimos `info()` y `describe(include="all").T` para verificar su estructura después de las transformaciones y eliminaciones.

In [21]:
# Definimos la ruta del dataset final generado en el bloque anterior.
RUTA_FINAL = Path("data_sets_transformados/secondary_market_modelo_final.csv")

# Cargamos el dataset final para revisar su estructura.
df_modelo_final = pd.read_csv(RUTA_FINAL)

# Mostramos las dimensiones finales del dataset.
print(f"Filas: {df_modelo_final.shape[0]} | Columnas: {df_modelo_final.shape[1]}")

# Revisamos tipos de datos y cantidad de valores no nulos por columna.
df_modelo_final.info()

# Calculamos la descripción estadística de las variables finales.
descripcion_final = df_modelo_final.describe(include="all").T

# Mostramos la descripción transpuesta para leer cada variable como una fila.
display(descripcion_final)

# Verificamos la cantidad total de valores faltantes restantes.
print(f"Valores faltantes totales: {df_modelo_final.isna().sum().sum()}")

Filas: 50000 | Columnas: 264
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Columns: 264 entries, lat to metro_distance_type_walk
dtypes: float64(8), int64(256)
memory usage: 100.7 MB


,count,mean,std,min,25%,50%,75%,max
lat,50000.0,55.712683,0.143998,55.23408,55.635415,55.74011,55.81749,56.02047
lon,50000.0,37.543657,0.180338,36.69923,37.467110,37.56584,37.65202,37.91986
total_area,50000.0,67.053158,17.309579,20.90000,54.400000,65.10000,77.50000,134.30000
living_area,50000.0,41.883782,11.209233,12.60000,33.700000,40.60000,48.50000,89.20000
kitchen_area,50000.0,11.244650,2.855538,6.00000,9.200000,11.10000,13.10000,24.80000
...,...,...,...,...,...,...,...,...
metro_station_Yasenevo,50000.0,0.018280,0.133963,0.00000,0.000000,0.00000,0.00000,1.00000
metro_station_Yugo-Zapadnaya,50000.0,0.012240,0.109956,0.00000,0.000000,0.00000,0.00000,1.00000
metro_station_Yuzhnaya,50000.0,0.011400,0.106162,0.00000,0.000000,0.00000,0.00000,1.00000
metro_distance_type_transport,50000.0,0.795420,0.403398,0.00000,1.000000,1.00000,1.00000,1.00000


Valores faltantes totales: 0


### Interpretación de la revisión final

Esta revisión permite confirmar que el dataset final conserva las variables transformadas, que las columnas solicitadas fueron eliminadas y que no aparecieron valores faltantes durante el proceso.

## 2.14 Análisis detallado del dataset final

### Paso 1: estructura con `info()`

Cargamos nuevamente el dataset final y mostramos explícitamente `info()` para revisar cantidad de filas, columnas, tipos de datos y valores no nulos.

In [22]:
# Definimos la ruta del dataset final.
RUTA_FINAL = Path("data_sets_transformados/secondary_market_modelo_final.csv")

# Cargamos el dataset final para analizar su estructura.
df_modelo_final = pd.read_csv(RUTA_FINAL)

# Mostramos las dimensiones generales del dataset.
print(f"Filas: {df_modelo_final.shape[0]} | Columnas: {df_modelo_final.shape[1]}")

# Mostramos explícitamente la información completa del DataFrame.
df_modelo_final.info()

# Confirmamos la cantidad total de valores faltantes después de las transformaciones.
print(f"Valores faltantes totales: {df_modelo_final.isna().sum().sum()}")

Filas: 50000 | Columnas: 264
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Columns: 264 entries, lat to metro_distance_type_walk
dtypes: float64(8), int64(256)
memory usage: 100.7 MB
Valores faltantes totales: 0


### Análisis de la estructura final

El dataset final contiene 50.000 registros y 264 columnas. Se retiraron las 12 columnas originales solicitadas y `price_rub` para evitar fuga de información hacia `price_per_sqm`; se conservaron las variables derivadas y las columnas one-hot. Las 264 columnas son numéricas: 256 enteras y 8 decimales. No quedan valores faltantes.

### Paso 2: `describe()` de las primeras 14 columnas

En este paso analizamos únicamente las primeras 14 columnas del dataset final.

In [23]:
# Seleccionamos las primeras 14 columnas respetando el orden del dataset final.
primeras_14_columnas = df_modelo_final.iloc[:, :14]

# Mostramos los nombres de las columnas seleccionadas.
print("Primeras 14 columnas:")
print(primeras_14_columnas.columns.tolist())

# Calculamos y mostramos sus estadísticas descriptivas transpuestas.
describe_primeras_14 = primeras_14_columnas.describe().T.round(2)
display(describe_primeras_14)

Primeras 14 columnas:
['lat', 'lon', 'total_area', 'living_area', 'kitchen_area', 'rooms', 'floor', 'total_floors', 'ceiling_height', 'metro_distance_min', 'to_center_km', 'price_per_sqm', 'mortgage_rate_at_listing', 'date_posted_year']


,count,mean,std,min,25%,50%,75%,max
lat,50000.0,55.71,0.14,55.23,55.64,55.74,55.82,56.02
lon,50000.0,37.54,0.18,36.70,37.47,37.57,37.65,37.92
total_area,50000.0,67.05,17.31,20.90,54.40,65.10,77.50,134.30
living_area,50000.0,41.88,11.21,12.60,33.70,40.60,48.50,89.20
kitchen_area,50000.0,11.24,2.86,6.00,9.20,11.10,13.10,24.80
rooms,50000.0,2.13,1.02,1.00,1.00,2.00,3.00,5.00
floor,50000.0,7.48,5.53,1.00,3.00,6.00,10.00,25.00
total_floors,50000.0,14.00,6.53,5.00,9.00,12.00,22.00,25.00
ceiling_height,50000.0,2.79,0.22,2.50,2.70,2.70,3.00,3.50
metro_distance_min,50000.0,57.06,54.48,2.00,22.00,39.00,71.00,328.00


### Análisis de las primeras 14 columnas

Las primeras 14 columnas corresponden a coordenadas, superficies, características del edificio, distancias, `price_per_sqm`, tasa hipotecaria y año de publicación. Los decimales son esperables porque representan coordenadas, superficies, distancias, tasas y estadísticas calculadas como media y desvío estándar.

En promedio, los inmuebles tienen 67,05 m² totales, 41,88 m² habitables y 11,24 m² de cocina. La mediana de ambientes es 2 y la de piso es 6. La distancia al metro tiene una media de 57,06 minutos y una mediana de 39, lo que indica una distribución sesgada por propiedades más alejadas.

Para `price_per_sqm`, la media es 195.625,67 RUB/m² y la mediana 170.400,00 RUB/m²; la diferencia confirma la influencia de valores altos. `price_rub` ya no está entre los predictores. El redondeo aplicado solo mejora la lectura de la tabla y no modifica los datos originales.